## Fetcher Tests

#### Manual Tests
These tests print the outputs of fetch_prices and fetch_ohlcv. Compare them against Yahoo Finance's own page.

In [22]:
import sys
sys.path.append('..')

from data.fetcher import fetch_prices, fetch_ohlcv

# fetch_prices: single ticker
prices = fetch_prices('AAPL', start='2023-01-01', end='2023-06-01')
assert list(prices.columns) == ['AAPL']
assert not prices.empty
prices.head()

Ticker,AAPL
Date,
2023-01-03,122.876747
2023-01-04,124.144127
2023-01-05,122.827621
2023-01-06,127.346947
2023-01-09,127.867630


In [23]:
# fetch_prices: multiple tickers
multi = fetch_prices(['AAPL', 'MSFT'], start='2023-01-01', end='2023-06-01')
assert set(multi.columns) == {'AAPL', 'MSFT'}
assert not multi.empty
multi.head()

Ticker,AAPL,MSFT
Date,,
2023-01-03,122.876747,232.510574
2023-01-04,124.144127,222.339783
2023-01-05,122.827621,215.750122
2023-01-06,127.346947,218.292862
2023-01-09,127.867630,220.418198


In [24]:
# fetch_ohlcv: full OHLCV for one ticker
ohlcv = fetch_ohlcv('AAPL', start='2023-01-01', end='2023-06-01')
assert {'Open', 'High', 'Low', 'Close', 'Volume'} <= set(ohlcv.columns)
assert not ohlcv.empty
ohlcv.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2023-01-03,122.876747,128.604505,121.992528,127.995383,112117500
2023-01-04,124.144127,126.403797,122.886574,124.664832,89113600
2023-01-05,122.827621,125.529397,122.572186,124.900621,80962700
2023-01-06,127.346947,128.005196,122.699897,123.800259,87754700
2023-01-09,127.867630,131.070471,127.612195,128.182026,70790800


#### Cross-check against Yahoo's own history page

Scrapes the rendered HTML table at `finance.yahoo.com/quote/{ticker}/history` directly —
an independent path from the JSON API `yfinance` calls — and compares it to
`fetch_ohlcv(..., auto_adjust=False)`. Automated but brittle (breaks if Yahoo changes their markup).

In [25]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


def scrape_yahoo_history(ticker: str, period1: int, period2: int) -> pd.DataFrame:
    """
    Parse Yahoo Finance's own history page table (not the API yfinance calls).
    period1/period2 are unix timestamps (seconds), as used in the page's URL.
    Raises RuntimeError if the page doesn't match the expected shape -- the
    simplest signal that Yahoo has changed their markup.
    """
    url = (f'https://finance.yahoo.com/quote/{ticker}/history/'
           f'?period1={period1}&period2={period2}')
    resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, 'lxml')

    table = soup.find('table')
    if table is None:
        raise RuntimeError('no <table> found on the page -- Yahoo may have changed their markup')

    tbody = table.find('tbody')
    if tbody is None:
        raise RuntimeError('table has no <tbody> -- Yahoo may have changed their markup')

    rows = tbody.find_all('tr')
    if not rows:
        raise RuntimeError('table has no rows -- Yahoo may have changed their markup')

    cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    all_records = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in rows]
    records = [r for r in all_records if len(r) == len(cols)]  # drop dividend/split rows

    dropped = len(all_records) - len(records)
    if not records or dropped > len(all_records) * 0.5:
        raise RuntimeError(
            f'{dropped}/{len(all_records)} rows did not have the expected {len(cols)} '
            f'columns (Date/Open/High/Low/Close/Adj Close/Volume) -- dividend rows '
            f'normally account for only a handful, so this looks like a markup change')

    df = pd.DataFrame(records, columns=cols)
    try:
        df['Date'] = pd.to_datetime(df['Date'], format='%b %d, %Y')
        for c in cols[1:-1]:
            df[c] = df[c].astype(float)
        df['Volume'] = df['Volume'].str.replace(',', '').astype(int)
    except (ValueError, TypeError) as e:
        raise RuntimeError(
            f'could not parse scraped values as expected ({e}) -- '
            f'Yahoo may have changed their markup') from e

    return df.set_index('Date').sort_index()

In [26]:
# Jan 1 2023 - Jun 1 2023, same window as the URL this check is based on
scraped = scrape_yahoo_history('AAPL', period1=1672531200, period2=1685577600)
raw = fetch_ohlcv('AAPL', start='2023-01-01', end='2023-06-01', auto_adjust=False)

common = scraped.index.intersection(raw.index)
assert len(common) == len(scraped) == len(raw)

for col in ['Close', 'Adj Close', 'Volume']:
    diff = (scraped.loc[common, col] - raw.loc[common, col]).abs()
    print(f'{col}: max diff = {diff.max()}')
    assert diff.max() < 0.01, f'{col} mismatch between scraped page and yfinance'

print('yfinance matches Yahoo\'s own history page for all', len(common), 'trading days')

Close: max diff = 7.32421875682121e-06
Adj Close: max diff = 0.004940185546871589
Volume: max diff = 0
yfinance matches Yahoo's own history page for all 103 trading days


## Value at Risk

Value at Risk (VaR) estimates the maximum expected loss over a time horizon at a given confidence level. It is typically computed by the historical method, variance-covariance method, and/or Monte Carlo simulation.


#### Historical Method
The historical method utilizes past data to produce an estimate for VaR. Rather than any model, it gets a percentile of the historical data and uses the percentile as its estimate.

However, this value must be scaled by \sqrt{horizon_days} to extend a 1-day VaR estimate to a longer horizon. This is the standard "square-root-of-time" rule (Basel Committee on Banking Supervision, 1996), and it comes from how variance behaves under an i.i.d. assumption:

- If daily returns are i.i.d. with variance σ², the n-day return `r_1 + ... + r_n` has variance `n·σ²` (variances of independent variables add).
- So its standard deviation is `σ·sqrt(n)`, not `n·σ` — risk grows with the square root of time, not linearly.
- Scaling every observation in the empirical return distribution by `sqrt(n)` scales every percentile of that distribution by the same `sqrt(n)`, which is exactly what you'd want if the n-day distribution were just the 1-day distribution stretched out. Under a normal-returns assumption this is exact; for the empirical distribution here it's an approximation, since it reuses the 1-day distribution's *shape* rather than the actual shape of n-day returns.

**Caveat:** the i.i.d. assumption is the load-bearing part, and it's known to be wrong in ways that matter — daily returns show volatility clustering, i.e. autocorrelated variance (Engle, 1982), and fatter tails than a single scaled day would suggest (Mandelbrot, 1963). Over longer horizons this rule tends to *understate* real tail risk when returns can jump (Danielsson and Zigrand, 2006), though it can also *overstate* long-horizon volatility under mean-reverting, GARCH-type dynamics (Diebold et al., 1997) — the direction of the error depends on the true data-generating process. It's a widely used regulatory convention (Basel Committee on Banking Supervision, 1996), not a statistically precise one — treat scaled multi-day VaR as an approximation, and prefer resampling actual n-day returns directly when the horizon is long or precision matters.

##### Historical VaR on Long Windows

In [ ]:
from models.risk import historical_var

# Fetch as much history as Yahoo has -- the backtests below need many
# independent blocks (especially at the 1-year horizon) to mean anything,
# and the short Jan-Jun 2023 window from the fetcher demo above isn't enough.
backtest_prices = fetch_prices('AAPL', start='1970-01-01')
returns = backtest_prices['AAPL'].pct_change().dropna()

var_95_1d = historical_var(returns, confidence=0.95, horizon_days=1)
var_95_10d = historical_var(returns, confidence=0.95, horizon_days=10)
# 252 trading days, matching the annualisation convention used in volatility.py --
# not 365, since `returns` is trading-day (not calendar-day) returns
var_95_1y = historical_var(returns, confidence=0.95, horizon_days=252)

print(f'({len(returns)} trading days, {returns.index.min().date()} to {returns.index.max().date()})')
print(f'1-day 95% historical VaR: {var_95_1d:.4%}')
print(f'10-day 95% historical VaR: {var_95_10d:.4%}')
print(f'1-year 95% historical VaR: {var_95_1y:.4%}')

In [ ]:
import numpy as np

# Backtest: what fraction of actual return windows breached each VaR?
# A well-calibrated 95% VaR should be exceeded ~5% of the time.
# Non-overlapping blocks, not a rolling window -- a rolling sum re-counts the
# same underlying bad stretch in every window it touches, which inflates the
# apparent sample size and can flip the conclusion.
exceed_1d = (returns <= var_95_1d).mean()
print(f'1-day: {exceed_1d:.2%} of {len(returns)} days exceeded VaR (target ~5.00%)')

blocks_10d = returns.groupby(np.arange(len(returns)) // 10).sum()
exceed_10d = (blocks_10d <= var_95_10d).mean()
print(f'10-day: {exceed_10d:.2%} of {len(blocks_10d)} non-overlapping 10-day blocks '
      f'exceeded VaR (target ~5.00%)')

blocks_1y = returns.groupby(np.arange(len(returns)) // 252).sum()
if len(blocks_1y) < 2:
    print(f'1-year: not enough data for a meaningful backtest -- {len(returns)} trading '
          f'days give only {len(blocks_1y)} non-overlapping 252-day block(s); need several '
          f'years of history for this to mean anything')
else:
    exceed_1y = (blocks_1y <= var_95_1y).mean()
    print(f'1-year: {exceed_1y:.2%} of {len(blocks_1y)} non-overlapping 1-year blocks '
          f'exceeded VaR (target ~5.00%)')

**References:**
- Basel Committee on Banking Supervision (1996), [*Amendment to the Capital Accord to Incorporate Market Risks*](https://www.bis.org/publ/bcbs23.pdf) — the regulatory source of the 1-day-to-10-day VaR scaling convention (§B.4, using sqrt(10)).
- Danielsson, J. and Zigrand, J-P. (2006), "On Time-Scaling of Risk and the Square-Root-of-Time Rule," *Journal of Banking & Finance*, 30(10), 2701-2713 — shows the rule systematically *understates* risk when returns can jump, worsening with horizon length.
- Diebold, F.X., Hickman, A., Inoue, A., and Schuermann, T. (1997), "Converting 1-Day Volatility to h-Day Volatility: Scaling by Root-h is Worse Than You Think," Wharton Financial Institutions Center Working Paper 97-34 (condensed as "Scale Models," *Risk*, 11, 104-107, 1998) — shows the same rule *overstates* long-horizon volatility variability under mean-reverting (GARCH-type) volatility dynamics.
- Engle, R.F. (1982), "Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation," *Econometrica*, 50(4), 987-1008 — the original ARCH model formalizing volatility clustering (the i.i.d.-variance violation above).
- Mandelbrot, B. (1963), "The Variation of Certain Speculative Prices," *Journal of Business*, 36(4), 394-419 — original empirical evidence that financial returns are fat-tailed relative to the normal distribution.